# Week 07 — Home exercise 2: The receipt, vectorized

**Solution proposal.**

The receipt from Part 1, written a third time — with NumPy arrays and no loop anywhere.

In [1]:
import numpy as np

items = ["Espresso machine", "Coffee beans", "Oat milk", "Filter papers"]
quantities = [1, 2, 3, 4]
unit_prices = [4999.00, 149.90, 24.50, 39.00]

VAT_RATE = 0.25

## 1. Lists become arrays

`items` stays a list: it is text, and we are not doing arithmetic on it.

In [2]:
quantity_array = np.array(quantities)
price_array = np.array(unit_prices)

print("quantities:", quantity_array, quantity_array.dtype)
print("prices:    ", price_array, price_array.dtype)

quantities: [1 2 3 4] int64
prices:     [4999.   149.9   24.5   39. ] float64


## 2. Every line total, in one expression

The two arrays are multiplied element by element: position 0 with position 0, position 1 with
position 1, and so on. No loop, no accumulator, no index.

In [3]:
line_totals = quantity_array * price_array

print(line_totals)

[4999.   299.8   73.5  156. ]


## 3. The totals

`.sum()` is a method on the array itself.

In [4]:
subtotal = line_totals.sum()
vat = subtotal * VAT_RATE
total = subtotal + vat

print(f"subtotal: {subtotal:>10,.2f}")
print(f"VAT:      {vat:>10,.2f}")
print(f"total:    {total:>10,.2f}")

subtotal:   5,528.30
VAT:        1,382.08
total:      6,910.38


## 4. A boolean mask, and what you can do with one

The comparison gives one `True` or `False` per item. Summing it counts the `True` values, because
`True` is 1 and `False` is 0. Using it inside square brackets keeps only the elements where it was
`True`.

In [5]:
expensive = line_totals > 100

print("above 100:", expensive)
print("how many: ", expensive.sum())
print(f"share of subtotal: {line_totals[expensive].sum() / subtotal:.1%}")

above 100: [ True  True False  True]
how many:  3
share of subtotal: 98.7%


## 5. The receipt

`zip` is from Part 1 and still the right tool here, because printing is not arithmetic.

In [6]:
for item, line_total in zip(items, line_totals.round(2)):
    print(f"{item:<20}{line_total:>10.2f}")

Espresso machine       4999.00
Coffee beans            299.80
Oat milk                 73.50
Filter papers           156.00


## What the loop was doing that this does not have to

The loop version had to carry three things the array version never mentions: an index or a `zip` to
keep the two lists in step, an accumulator initialized before the loop, and an update inside it. Each
of those is a place to make a mistake, and the accumulator in particular has to be in exactly the
right place — initialize it inside the loop and you get the last item's value.

The array version says what it wants rather than how to get it: "quantities times prices" is one
expression, and "the sum of those" is one more. There is nothing to keep in step, because the arrays
already are.

That is the real argument for vectorization, and it is not about speed. Speed is the argument at a
million rows; at four rows the argument is that there are fewer places to be wrong.

### Things worth noticing

- `line_totals[expensive]` is the same idea as filtering a DataFrame with a condition. A boolean mask
  used inside square brackets keeps the `True` positions. When you write `co2[co2["year"] == 2023]`,
  this is what is happening one level down.
- `.round(2)` returns a **new** array rather than rounding in place, which is the same convention
  pandas follows and the opposite of what list methods do.
- The share comes out at 98.7%, which sounds impressive until you notice that only one of the four
  lines is below the threshold, and that the espresso machine alone is 90% of the receipt. A
  percentage computed over four items is arithmetic, not evidence.

### What this program does NOT do

- It does not check that the two arrays are the same length. Multiplying arrays of different lengths
  raises a `ValueError` rather than doing something silently wrong, which is better than `zip`'s
  behavior of stopping at the shortest — but it is still an error you have to read.
- It keeps the item names in a plain list beside the arrays, so the names and the numbers are still
  only related by position. That is the same fragility the loop version had, and the fix is a
  DataFrame, where the column names and the values travel together.